In [ ]:
# ==============================================================================
# CELL 1 / PHASE 1: UNIVERSAL DECIMAL CLASSIFICATION (UDC) INGESTION
# 
# ARCHITECTURE NOTE: Because the official UDC linked data endpoint is offline 
# for revision, and the fallback PHP summary site relies on dynamic JS trees 
# that are brittle to scrape, this script utilizes Strategy A (Local Bulk Parsing).
#
# PREREQUISITE: Ensure 'udcs-skos.rdf' (sourced from an open-source mirror
# or archive) is placed in the 'data/external/' directory before execution.
# ==============================================================================

import os
import sys
import re
import pandas as pd
from rdflib import Graph, URIRef
from rdflib.namespace import SKOS, RDF
from collections import deque
from dotenv import load_dotenv

# --- 1. Environment & Central Schema Setup ---
load_dotenv("../../config/.env") 

notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
raw_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "raw"))
external_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "external"))
config_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "config"))

os.makedirs(raw_data_dir, exist_ok=True)
os.makedirs(external_data_dir, exist_ok=True)

sys.path.append(config_dir)
try:
    from ingest_schema_manager import get_registry_info, finalize_row, COLUMN_ORDER
except ImportError:
    raise ImportError("CRITICAL: Could not find ingest_schema_manager.py. Check PYTHONPATH.")

# --- 2. Registry Lookup (UDC) ---
SOURCE_PREFIX = "UDC"
registry_data = get_registry_info(
    prefix=SOURCE_PREFIX, 
    config_dir=config_dir, 
    fallback_name="Universal Decimal Classification Summary", 
    fallback_uri="http://udcdata.info/"
)

SOURCE_NAME = registry_data['Source_Name']
raw_ingest_file = os.path.join(raw_data_dir, f"raw_{SOURCE_PREFIX.lower()}.csv")
local_rdf_path = os.path.join(external_data_dir, "udcs-skos.rdf")

if not os.path.exists(local_rdf_path):
    print(f"[!] CRITICAL ERROR: Could not find {local_rdf_path}.")
    print("Please place the UDC Summary SKOS file in data/external/.")
    sys.exit(1)

# --- 3. Load Graph ---
print("Loading UDC RDF graph into local memory...")
g = Graph()
g.parse(local_rdf_path, format="xml")
print(f"Graph loaded successfully. Total triples: {len(g)}")

# Target seeds: 
TARGET_SEEDS = [
    URIRef("http://udcdata.info/015988"), # Religion and Theology root
    URIRef("http://udcdata.info/015237"), # The paranormal. The occult. Psi phenomena
    URIRef("http://udcdata.info/023314"), # Ecclesiastical law. Canon law. Religious law (option)
    URIRef("http://udcdata.info/019301"), # Relations between church and state. Policy towards religion. Church policy
    ]

# --- 4. BFS Ancestry Resolution ---
# BFS mathematically guarantees the shortest path to the root, preventing 
# circuitous drift if any lateral links are miscoded as hierarchical in the RDF.
path_cache = {}

def get_english_label(uri):
    """Helper to extract strictly the English skos:prefLabel."""
    for obj in g.objects(uri, SKOS.prefLabel):
        lang = getattr(obj, 'language', '')
        if lang and lang.startswith('en'):
            return re.sub(r'\s+', ' ', str(obj)).strip()
    return str(uri).split('/')[-1]

def get_best_path(uri):
    """Uses Breadth-First Search to construct the Hierarchy_Path upward."""
    uri_str = str(uri)
    if uri_str in path_cache:
        return path_cache[uri_str]

    queue = deque([(uri, [])])
    visited = set([uri])
    
    while queue:
        current_node, current_path = queue.popleft()
        
        parents = list(g.objects(current_node, SKOS.broader))
        
        # If we hit an absolute root node
        if not parents:
            full_path_uris = [current_node] + current_path
            breadcrumbs = [get_english_label(u) for u in full_path_uris]
            final_path_str = " > ".join(breadcrumbs)
            path_cache[uri_str] = final_path_str
            return final_path_str
            
        for p in parents:
            if p not in visited:
                visited.add(p)
                queue.append((p, [current_node] + current_path))
                
    fallback_label = get_english_label(uri)
    path_cache[uri_str] = fallback_label
    return fallback_label

# --- 5. Extraction Logic ---
visited_uris = set()
rows_to_export = []

def process_node(uri):
    """Recursively parses downward to extract all narrower concepts."""
    uri_str = str(uri)
    if uri_str in visited_uris:
        return
    visited_uris.add(uri_str)
    
    # 1. Dynamic Concept Type Extraction
    concept_type = ""
    for obj in g.objects(uri, RDF.type):
        type_str = str(obj)
        if "skos/core#" in type_str:
            concept_type = f"skos:{type_str.split('#')[-1]}"
        elif "owl#" in type_str:
            concept_type = f"owl:{type_str.split('#')[-1]}"
        else:
            concept_type = type_str.split('/')[-1]
            
    if not concept_type:
        concept_type = "skos:Concept" # Safe fallback
    
    # 2. Label and Translation Tracking
    primary_label = ""
    has_translation = ""
    synonyms = []
    
    for obj in g.objects(uri, SKOS.prefLabel):
        lang = getattr(obj, 'language', '')
        if lang and lang.startswith('en'):
            primary_label = re.sub(r'\s+', ' ', str(obj)).strip()
        elif lang and not lang.startswith('en'):
            has_translation = "yes"
            
    for prop in [SKOS.altLabel, SKOS.hiddenLabel]:
        for obj in g.objects(uri, prop):
            lang = getattr(obj, 'language', '')
            if lang and lang.startswith('en'):
                clean_syn = re.sub(r'\s+', ' ', str(obj)).strip()
                if clean_syn not in synonyms:
                    synonyms.append(clean_syn)
            
    if not primary_label:
        primary_label = "No English Label"

    print(f"\rIngesting [{len(visited_uris)}]: {primary_label[:70]:<70}", end="", flush=True)

    # 3. Extract Standard Decimal Notation and route to Synonyms
    for obj in g.objects(uri, SKOS.notation):
        decimal_notation = str(obj).strip()
        if decimal_notation and decimal_notation not in synonyms:
            synonyms.append(decimal_notation)
        break 

    # 4. Compile Descriptions (Scope notes and standard notes)
    descriptions = []
    for prop in [SKOS.scopeNote, SKOS.note]:
        for obj in g.objects(uri, prop):
            if getattr(obj, 'language', '').startswith('en') or not getattr(obj, 'language', None):
                clean_desc = re.sub(r'\s+', ' ', str(obj)).strip()
                descriptions.append(clean_desc)
                
    # 5. Resolve Parents
    parent_ids = []
    for p in g.objects(uri, SKOS.broader):
        p_id = str(p).split('/')[-1]
        parent_ids.append(p_id)
        
    # 6. Extract Native Crosswalks
    crosswalks = []
    for prop in [SKOS.exactMatch, SKOS.closeMatch]:
        for obj in g.objects(uri, prop):
            crosswalks.append(str(obj))

    # 7. Map to the 15-Column Schema
    native_integer = uri_str.split('/')[-1]

    extracted_data = {
        'Source_System': SOURCE_NAME,
        'Primary_Label': primary_label,
        'CURIE': f"{SOURCE_PREFIX}:{native_integer}", 
        'Formal_Label': "",  # Intentionally blank per source methodology
        'Concept_Type': concept_type,
        'Hierarchy_Path': get_best_path(uri),
        'Synonyms': " | ".join(synonyms),
        'Description': " | ".join(descriptions),
        'Parent_IDs': " | ".join(parent_ids),
        'Concept_ID': native_integer,
        'URI': uri_str,
        'Has_Translation': has_translation,
        'Status': 'active',
        'Crosswalks': " | ".join(crosswalks)
    }
    
    rows_to_export.append(finalize_row(extracted_data))
    
    # 8. Recurse Downward
    for child in g.objects(uri, SKOS.narrower):
        process_node(child)

# --- 6. Execution ---
print(f"\nStarting UDC extraction from {len(TARGET_SEEDS)} root seed(s)...\n")
for seed in TARGET_SEEDS:
    process_node(seed)

# --- 7. Final Output Generation ---
print(f"\n\nFormatting {len(rows_to_export)} concepts into Bronze schema...")
df_final = pd.DataFrame(rows_to_export)[COLUMN_ORDER]
df_final.to_csv(raw_ingest_file, index=False, encoding='utf-8-sig')

print(f"Success! Data exported to: {raw_ingest_file}")

In [ ]:
# ==============================================================================
# CELL 2: LATERAL DISCOVERY & SEED VALIDATION (UDC)
# 
# ARCHITECTURE NOTE: This script utilizes the local RDF file (Strategy A) to 
# instantly query lateral 'skos:related' links connected to our captured baseline.
# It deduplicates these against the existing Bronze Layer and exports net-new 
# concepts to an ephemeral "Inbox" CSV for human review and potential seed expansion.
# ==============================================================================

import os
import sys
import re
import pandas as pd
from rdflib import Graph, URIRef
from rdflib.namespace import SKOS
from collections import deque
from dotenv import load_dotenv

RUN_LATERAL_DISCOVERY = False  # Toggle to True to execute

if RUN_LATERAL_DISCOVERY:
    # --- 1. Setup Paths & Environment ---
    load_dotenv("../../config/.env") 
    
    notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
    raw_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "raw"))
    external_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "external"))
    
    raw_udc_file = os.path.join(raw_data_dir, "raw_udc.csv")
    output_candidates_file = os.path.join(raw_data_dir, "udc_lateral_candidates.csv")
    local_rdf_path = os.path.join(external_data_dir, "udcs-skos.rdf")
    
    # --- 2. Load Existing Concepts (Deduplication Baseline) ---
    if not os.path.exists(raw_udc_file):
        raise FileNotFoundError(f"[!] CRITICAL: Could not find {raw_udc_file}. Please run Cell 1 first.")
        
    existing_df = pd.read_csv(raw_udc_file, dtype={'Concept_ID': str})
    existing_uris = set(existing_df['URI'].dropna().astype(str).str.strip())
    print(f"[*] Loaded {len(existing_uris)} existing UDC concepts to serve as the baseline.")
    
    # --- 3. Load Local Graph ---
    if not os.path.exists(local_rdf_path):
        raise FileNotFoundError(f"[!] CRITICAL: Could not find {local_rdf_path}.")
        
    print("[*] Loading UDC RDF graph into local memory...")
    g = Graph()
    g.parse(local_rdf_path, format="xml")
    
    # --- 4. Bottom-Up Ancestry Function (Duplicated for Cell Independence) ---
    path_cache = {}
    
    def get_english_label(uri):
        """Helper to extract strictly the English skos:prefLabel."""
        for obj in g.objects(uri, SKOS.prefLabel):
            lang = getattr(obj, 'language', '')
            if lang and lang.startswith('en'):
                return re.sub(r'\s+', ' ', str(obj)).strip()
        return str(uri).split('/')[-1]

    def get_best_path(uri):
        """Uses Breadth-First Search to construct the absolute Hierarchy_Path."""
        uri_str = str(uri)
        if uri_str in path_cache:
            return path_cache[uri_str]

        queue = deque([(uri, [])])
        visited = set([uri])
        
        while queue:
            current_node, current_path = queue.popleft()
            parents = list(g.objects(current_node, SKOS.broader))
            
            if not parents:
                full_path_uris = [current_node] + current_path
                breadcrumbs = [get_english_label(u) for u in full_path_uris]
                final_path_str = " > ".join(breadcrumbs)
                path_cache[uri_str] = final_path_str
                return final_path_str
                
            for p in parents:
                if p not in visited:
                    visited.add(p)
                    queue.append((p, [current_node] + current_path))
                    
        fallback_label = get_english_label(uri)
        path_cache[uri_str] = fallback_label
        return fallback_label

    # --- 5. Harvest Lateral Concepts ---
    candidates_data = []
    candidates_seen = set() # To prevent duplicates within the candidate list itself
    
    print("\n[*] Scanning local RDF for lateral associations...")
    
    for i, uri_str in enumerate(list(existing_uris), 1):
        uri = URIRef(uri_str)
        
        print(f"\rScanning [{i}/{len(existing_uris)}]: {uri_str.split('/')[-1]:<15}", end="", flush=True)
        
        # Query all skos:related edges connected to this ingested concept
        for related_node in g.objects(uri, SKOS.related):
            r_uri = str(related_node).strip()
            
            # Deduplicate against baseline AND current candidate list
            if r_uri not in existing_uris and r_uri not in candidates_seen:
                candidates_seen.add(r_uri)
                
                # Fetch semantic context for the human reviewer
                candidate_path = get_best_path(related_node)
                candidate_label = get_english_label(related_node)
                
                candidates_data.append({
                    'Candidate_URI': r_uri, 
                    'Candidate_Label': candidate_label,
                    'Hierarchy_Path': candidate_path,
                    'Suggested_By_URI': uri_str # Tracking provenance
                })
                
    # --- 6. Export and Sort ---
    print(f"\n\nLateral Discovery Complete! Found {len(candidates_data)} unique candidates outside current boundaries.")
    
    if candidates_data:
        candidates_df = pd.DataFrame(candidates_data)
        
        # Sort by hierarchy so entire branches group together visually in the CSV
        candidates_df = candidates_df.sort_values(by='Hierarchy_Path')
        candidates_df.to_csv(output_candidates_file, index=False, encoding='utf-8-sig')
        
        print(f"[*] Sorted candidates saved to: {output_candidates_file}")
        print("\nNext Step: Open this CSV, review the Hierarchy_Path column, and identify any missing parent branches to add as TARGET_SEEDS in Cell 1.")
    else:
        print("[*] No new relevant concepts found. Your seed list is structurally bounded!")
else:
    print("Lateral Discovery is disabled. Set RUN_LATERAL_DISCOVERY = True to execute.")